# Práctica 2 · Bivariada y visualización

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Lautaromgo/estadistica-aplicada-unne/blob/main/notebooks/practica/practica_2_bivariada_visualizacion.ipynb)

En la práctica 1 describiste **una** variable por vez. Acá pasamos a **dos**, que es donde
empiezan las preguntas interesantes: *¿las dos se mueven juntas?*, *¿cuánto?*, *¿ese número
significa lo que parece?*.

Funciona igual que la anterior: cinco funciones vacías con un `TODO` adentro, y una celda de
verificación después de cada una. Las de la práctica 1 ya vienen resueltas en el setup, así
que podés apoyarte en ellas.

Corré todo de arriba hacia abajo. No hace falta instalar nada, acá ni en Colab.

## 0 · Setup

Trae el dataset, las dos derivadas que vamos a usar, `verificar()`, y `media()` y
`desvio_estandar()` de la práctica anterior — ya hechas.

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)

AZUL, ROJO, AZUL_PALE = "#1E579B", "#D1495B", "#DCEAF7"

URL = ("https://raw.githubusercontent.com/ageron/handson-ml2/"
       "master/datasets/housing/housing.csv")
RUTAS = ["../../data/raw/housing.csv", "../data/raw/housing.csv", "housing.csv"]

ruta = next((r for r in RUTAS if os.path.exists(r)), None)
df = pd.read_csv(ruta) if ruta else pd.read_csv(URL)

# "Cuánto hay POR HOGAR" en vez de "cuánto hay en el barrio": los totales del
# dataset dependen del tamaño del barrio, y eso tapa todo lo demás.
df["ambientes_por_hogar"] = df["total_rooms"] / df["households"]
df["personas_por_hogar"] = df["population"] / df["households"]


def verificar(nombre, obtenido, esperado, tol=1e-6):
    """Compara tu resultado con el de pandas. Avisa y sigue: no corta la notebook."""
    try:
        ok = bool(np.isclose(float(obtenido), float(esperado), rtol=tol, atol=tol))
    except (TypeError, ValueError):
        ok = False
    print(f"✅ {nombre}" if ok
          else f"❌ {nombre} → obtuviste {obtenido}, se esperaba {esperado}")


def nro(x, decimales=0):
    """Formatea un número como lo escribimos nosotros: 20.640 · 0,98"""
    return f"{x:,.{decimales}f}".replace(",", "|").replace(".", ",").replace("|", ".")


# --- De la práctica 1, ya resueltas ---------------------------------------
def media(valores):
    x = np.asarray(valores, dtype=float)
    x = x[~np.isnan(x)]
    return np.nan if x.size == 0 else x.sum() / x.size


def desvio_estandar(valores, muestra=True):
    x = np.asarray(valores, dtype=float)
    x = x[~np.isnan(x)]
    if x.size < 2:
        return np.nan
    return np.sqrt(((x - media(x)) ** 2).sum() / (x.size - 1 if muestra else x.size))


print(f"{nro(len(df))} filas × {df.shape[1]} columnas   |   origen: {ruta or 'la web'}")

## 1 · ¿Se mueven juntas? La covarianza

El par de esta práctica: **`ambientes_por_hogar`** y **`median_income`**. La hipótesis de
sentido común es que los barrios con ingresos más altos tienen casas más grandes. Vamos a ver
cuánto de eso está en los datos.

La idea de la covarianza es simple y vale la pena tenerla clara antes de escribir una línea:
para cada barrio miramos si los ambientes están **por encima o por debajo de su propia
media**, y si el ingreso está por encima o por debajo de **la suya**. Si los dos se desvían
para el mismo lado, el producto da positivo; si se desvían para lados opuestos, negativo. La
covarianza es el promedio de esos productos.

El error clásico es comparar los valores de `x` con los de `y`. No: cada uno con su propia
media. Son peras y manzanas hasta que los convertís en desvíos.

In [ ]:
def covarianza(x, y, muestra=True):
    """El promedio de los productos de los desvíos de cada variable respecto de SU media.

    Lo único que se lee de este número es EL SIGNO: positivo, las dos suben juntas;
    negativo, cuando una sube la otra baja. La intensidad no se puede leer acá, y la
    celda de más abajo muestra por qué.

    Las filas donde falte cualquiera de las dos no sirven para este cálculo: se descartan
    las dos juntas, no cada columna por su lado.
    """
    # ─── TODO ────────────────────────────────────────────────────────────
    #   1. Armá un DataFrame con las dos series y hacele .dropna(): si a una fila le falta
    #      cualquiera de las dos, esa fila no entra.
    #   2. Si quedan menos de 2 filas, devolvé np.nan.
    #   3. Calculá la media de cada columna POR SEPARADO (podés usar media()).
    #   4. Restale a cada valor la media DE SU PROPIA columna, multiplicá las dos diferencias
    #      fila por fila, y sumá.
    #   5. Dividí por (n - 1) si muestra=True, o por n si muestra=False.
    # ─────────────────────────────────────────────────────────────────────
    raise NotImplementedError("Falta completar covarianza(). Mirá el TODO justo arriba.")

In [ ]:
amb, ing = df["ambientes_por_hogar"], df["median_income"]

verificar("covarianza", covarianza(amb, ing), amb.cov(ing))
verificar("covarianza poblacional", covarianza(amb, ing, muestra=False),
          amb.cov(ing) * (len(df) - 1) / len(df))

#### ⚠️ Probalo y mirá (1): el mismo dato, dos covarianzas distintas

`median_income` está en decenas de miles de dólares. Pasalo a dólares —multiplicar por
10.000, que no cambia absolutamente nada de la realidad— y volvé a calcular.

In [ ]:
en_dolares = ing * 10_000

print(f"ingreso en decenas de miles →  cov = {nro(covarianza(amb, ing), 2):>12}")
print(f"ingreso en dólares          →  cov = {nro(covarianza(amb, en_dolares), 2):>12}")

El mismo par de columnas, los mismos barrios, la misma relación: **1,54** o **15.365,68**,
según en qué unidad esté guardado el ingreso.

La covarianza queda en "unidades de x multiplicadas por unidades de y", y eso no es una
magnitud que se pueda interpretar ni comparar contra otro par. Por eso de la covarianza se lee
el signo y nada más.

> **La regla:** si un número cambia cuando cambiás la unidad de medida, ese número no mide la
> intensidad de nada.

## 2 · La correlación: la covarianza, pero legible

La solución es dividir la covarianza por los dos desvíos. Ahí se cancelan las unidades y queda
un número entre **−1 y 1**, sin unidades, que **sí** se puede comparar entre pares distintos.
Eso es el coeficiente de correlación de Pearson, el famoso `r`.

In [ ]:
def correlacion(x, y):
    """La covarianza dividida por los dos desvíos: el coeficiente de Pearson, r.

    CÓMO SE LEE
        r = 0     no hay relación LINEAL
        r → 1     cuando una sube, la otra sube, y los puntos se acercan a una recta
        r → -1    lo mismo, pero al revés

    Y OJO CON ESTO, QUE ES LA TRAMPA MÁS CARA: Pearson mide relación LINEAL. Un r cercano a
    cero puede significar "no hay relación" o "hay una relación fuerte que no es una recta",
    y son cosas completamente distintas. Por eso nunca se reporta un r sin haber mirado el
    gráfico.
    """
    # ─── TODO ────────────────────────────────────────────────────────────
    #   1. Armá el par y sacale los nulos, igual que en covarianza().
    #   2. Calculá el desvío de cada columna con desvio_estandar().
    #   3. Si alguno de los dos desvíos es 0, devolvé np.nan: una columna constante no
    #      correlaciona con nada.
    #   4. Devolvé covarianza(x, y) dividido por el producto de los dos desvíos.
    # ─────────────────────────────────────────────────────────────────────
    raise NotImplementedError("Falta completar correlacion(). Mirá el TODO justo arriba.")

In [ ]:
verificar("correlación", correlacion(amb, ing), amb.corr(ing))

# Y la prueba que la covarianza no pasaba: cambiar la unidad no la mueve.
print(f"\nr con el ingreso en decenas de miles = {correlacion(amb, ing):.4f}")
print(f"r con el ingreso en dólares          = {correlacion(amb, en_dolares):.4f}")

#### ⚠️ Probalo y mirá (2): la correlación más alta del dataset no es ningún hallazgo

Pedile a `pandas` la matriz de correlación y ordená los pares de mayor a menor.

In [ ]:
matriz = df.select_dtypes("number").corr()

# Nos quedamos con la mitad de arriba para no repetir cada par dos veces.
pares = (matriz.where(np.triu(np.ones(matriz.shape), k=1).astype(bool))
               .stack()
               .sort_values(key=abs, ascending=False))

print(pares.head(5).round(3).to_string())

El podio es **`total_bedrooms` × `households` = 0,980**. Dicho así suena a descubrimiento.

Ahora pensá qué mide cada una: `total_bedrooms` son los dormitorios **del barrio entero** y
`households` son los hogares **del barrio entero**. Un barrio con el doble de hogares tiene el
doble de dormitorios, de habitaciones y de gente. Las dos columnas están midiendo, sobre todo,
**cuán grande es el barrio** — y por eso el `r` es altísimo y no dice nada que no supieras.

Eso es exactamente lo que arregla dividir por `households`: `ambientes_por_hogar` ya no depende
del tamaño del barrio, y su `r` con el ingreso (**0,33**) sí es una afirmación sobre el mundo.

> **La regla:** antes de festejar un `r` alto, preguntate qué mide cada columna. Dos formas de
> medir lo mismo siempre van a correlacionar.

## 3 · Mirarlo antes de creerlo

Un `r` es un resumen de 20.640 filas en un solo número, y todo resumen tira información a la
basura. El gráfico de dispersión es el control: cada punto es un barrio, y ahí se ve la forma
que el número no puede contar.

Dos decisiones que parecen cosméticas y no lo son:

- **`alpha`** — con miles de puntos superpuestos, donde hay 10 y donde hay 3.000 se ve igual
  de negro. Bajando la opacidad, la densidad vuelve a verse.
- **`muestra`** — dibujar 4.000 puntos al azar en vez de 20.640 no cambia la conclusión y sí
  cambia lo que se puede leer. El `random_state` va fijo para que el gráfico no cambie en cada
  corrida.

In [ ]:
def graficar_dispersion(df, columna_x, columna_y, muestra=4000, alpha=0.35, random_state=42):
    """Scatter de dos columnas numéricas, dibujado para que se pueda leer.

    Devuelve el `ax` de matplotlib, por si después querés agregarle algo encima.

    El título lleva cuántas filas se están dibujando de cuántas: quien mire el gráfico
    tiene que poder saber que no están todas.
    """
    # ─── TODO ────────────────────────────────────────────────────────────
    #   1. Sacá las filas donde falte alguna de las dos columnas:
    #        datos = df.dropna(subset=[columna_x, columna_y])
    #   2. Guardate cuántas filas quedaron (el total), y si son más que `muestra`,
    #      quedate con una muestra al azar: datos.sample(muestra, random_state=random_state)
    #   3. Creá la figura con fig, ax = plt.subplots(figsize=(9, 5)) y dibujá con
    #      ax.scatter(..., s=14, alpha=alpha, color=AZUL, edgecolor="none")
    #   4. Poné nombres a los ejes y un título que diga cuántas filas de cuántas estás dibujando.
    #   5. Devolvé ax.
    # ─────────────────────────────────────────────────────────────────────
    raise NotImplementedError("Falta completar graficar_dispersion(). Mirá el TODO justo arriba.")

In [ ]:
ax = graficar_dispersion(df, "ambientes_por_hogar", "median_income")
ax.set_xlim(0, 20)          # el eje completo llega a 142 y aplasta todo contra la izquierda
plt.tight_layout()
plt.show()

#### ⚠️ Probalo y mirá (3): 20.640 puntos son una mancha, y 21 filas se comen un tercio del `r`

Primero, la misma nube dibujada de las dos formas. Después, el `r` calculado con y sin el
**0,1% más alto** de `ambientes_por_hogar` — las mismas 21 filas de la práctica anterior, las
que ni siquiera entran en el gráfico.

In [ ]:
fig, (izq, der) = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)

paneles = [(izq, df, 1.0, f"{nro(len(df))} puntos, alpha = 1"),
           (der, df.sample(4000, random_state=42), 0.35, "4.000 puntos, alpha = 0,35")]

for ax, datos, a, titulo in paneles:
    ax.scatter(datos["ambientes_por_hogar"], datos["median_income"],
               s=14, alpha=a, color=AZUL, edgecolor="none")
    ax.set_xlim(0, 20)
    ax.set_xlabel("ambientes_por_hogar")
    ax.set_title(titulo)

izq.set_ylabel("median_income")
plt.tight_layout()
plt.show()

corte = amb.quantile(0.999)
sin_extremos = df[df["ambientes_por_hogar"] <= corte]

print(f"r con las {nro(len(df))} filas          = {correlacion(amb, ing):.4f}")
print(f"r sin el 0,1% más alto ({(amb > corte).sum()} filas) = "
      f"{correlacion(sin_extremos['ambientes_por_hogar'], sin_extremos['median_income']):.4f}")

En el panel de la izquierda, la zona densa es un bloque sólido: no se distingue dónde viven
tres barrios y dónde viven tres mil. En el de la derecha, con menos puntos y menos opacidad,
aparece la forma: un cono que se abre: a más ambientes por hogar, más ingreso, pero con una
dispersión enorme. Y aparece también una cola de barrios pasados los 10 ambientes por hogar
que no sigue el patrón: mucho espacio y poco ingreso.

Y abajo, el golpe: **21 filas de 20.640** —el 0,1%, tan lejos que ni entran en el gráfico—
hunden el `r` de **0,48 a 0,33**. Un tercio de la correlación se lo come una milésima parte de
los datos.

> **La regla:** el gráfico no es la ilustración del análisis, es parte del análisis. Un `r`
> reportado sin haber mirado la nube es un número que no sabés de dónde viene.

## 4 · Cruzar dos categóricas

Con dos variables cualitativas no hay covarianza ni correlación que valga: lo que se hace es
contar cuántos casos caen en cada cruce. Eso es una **tabla de contingencia**.

Para tener una segunda categórica, partimos el ingreso en tramos. Ese corte es una decisión
—dónde ponés los límites cambia la tabla— y por eso va a la vista, en la misma celda.

In [ ]:
def tabla_cruzada(df, columna_1, columna_2):
    """Tabla de contingencia normalizada POR FILA, con la columna n al lado.

    POR QUÉ NORMALIZAR: los conteos crudos te dicen qué grupo es más GRANDE, no cómo se
    COMPORTA cada grupo. Con cada fila sumando 100%, grupos de tamaños muy distintos se
    vuelven comparables.

    POR QUÉ LA COLUMNA n: un porcentaje esconde cuántos casos hay detrás. Un 100% sobre
    cinco filas y un 100% sobre cinco mil se escriben igual y no valen lo mismo.
    """
    # ─── TODO ────────────────────────────────────────────────────────────
    #   1. Contá los cruces: conteos = pd.crosstab(df[columna_1], df[columna_2])
    #   2. Volvé a llamar a pd.crosstab con normalize="index" y multiplicá por 100.
    #      Así cada FILA suma 100%. Redondeá a 1 decimal.
    #   3. Agregale la columna "n" con el total de cada fila de los conteos crudos:
    #        conteos.sum(axis=1)
    #   4. Devolvé la tabla.
    # ─────────────────────────────────────────────────────────────────────
    raise NotImplementedError("Falta completar tabla_cruzada(). Mirá el TODO justo arriba.")

In [ ]:
# El corte es una decisión nuestra, no del dataset. En decenas de miles de USD:
# hasta 25.000, entre 25.000 y 45.000, y de ahí para arriba.
df["tramo_ingreso"] = pd.cut(df["median_income"], bins=[0, 2.5, 4.5, np.inf],
                             labels=["bajo", "medio", "alto"])

t = tabla_cruzada(df, "ocean_proximity", "tramo_ingreso")

esperado = pd.crosstab(df["ocean_proximity"], df["tramo_ingreso"],
                       normalize="index").loc["INLAND", "alto"] * 100
verificar("% de INLAND en el tramo alto", t.loc["INLAND", "alto"], round(esperado, 1))
verificar("n de ISLAND", t.loc["ISLAND", "n"], 5)

print()
t

**Tierra adentro, el 15,9% de los barrios está en el tramo alto de ingresos. A menos de una
hora del océano, el 36,8%** — más del doble. Esa es la clase de frase que la tabla cruzada
permite decir, y que los conteos crudos no: `INLAND` tiene *más* barrios de ingreso alto en
números absolutos (1.042 contra ninguno de `ISLAND`) simplemente porque tiene 6.551 filas.

Y ahí está `ISLAND` otra vez: **80% en el tramo medio**, que suena contundente hasta que leés
la columna `n`. Son **cinco barrios**: el 80% son cuatro casas.

> **La regla:** un porcentaje sin el `n` al lado no es un resultado, es una opinión con
> formato de número.

## 5 · Los que se van lejos del centro

Último ejercicio. La regla más usada para marcar valores atípicos usa los cuartiles que ya
conocés: se mide el rango intercuartílico `IQR = Q3 − Q1` y se marca todo lo que caiga a más de
`k` veces el IQR por fuera de la caja.

El `k = 1,5` que trae todo el mundo por defecto es una **convención**, no un resultado.

In [ ]:
def detectar_outliers(df, columna, k=1.5, devolver_limites=False):
    """Marca las filas cuyo valor cae a más de k·IQR por fuera de los cuartiles.

    Devuelve las filas atípicas. Con devolver_limites=True, la tupla
    (filas, lim_inferior, lim_superior).

    LO QUE ESTA REGLA NO PUEDE VER: mira una columna por vez. Hay filas absurdas cuyos
    valores son normales en cada columna por separado y sólo son imposibles como
    combinación. Para eso hay que mirar el par, no el valor.
    """
    # ─── TODO ────────────────────────────────────────────────────────────
    #   1. Sacá los nulos de la columna.
    #   2. Q1 y Q3 salen de np.percentile(valores, [25, 75]).
    #   3. IQR = Q3 - Q1, y los límites son  Q1 - k*IQR  y  Q3 + k*IQR.
    #   4. Quedate con las filas del df que caen FUERA de esos límites (menor que el inferior
    #      O mayor que el superior).
    #   5. Si devolver_limites=True, devolvé (filas, lim_inf, lim_sup); si no, sólo las filas.
    # ─────────────────────────────────────────────────────────────────────
    raise NotImplementedError("Falta completar detectar_outliers(). Mirá el TODO justo arriba.")

In [ ]:
v = df["personas_por_hogar"]
q1, q3 = v.quantile(0.25), v.quantile(0.75)
esperado = ((v < q1 - 1.5 * (q3 - q1)) | (v > q3 + 1.5 * (q3 - q1))).sum()
verificar("atípicos con k = 1,5", len(detectar_outliers(df, "personas_por_hogar")), esperado)

print()
for k in (1.5, 3.0):
    at, lo, hi = detectar_outliers(df, "personas_por_hogar", k=k, devolver_limites=True)
    print(f"k = {k}:  {len(at):>4} atípicos ({len(at)/len(df):>5.1%})   "
          f"límites [{lo:.2f}, {hi:.2f}]   el mayor: {at['personas_por_hogar'].max():.1f}")

Con `k = 1,5` marcás **711 barrios**; con `k = 3,0`, **132**. Cinco veces menos, y la única
diferencia entre las dos respuestas es un número que nadie justifica nunca.

Pero mirá la última columna: el máximo es el mismo en los dos casos. El barrio de **1.243
personas por hogar** lo marca cualquier `k`. Lo difícil no son los disparates: es todo lo del
medio, donde el umbral decide si un barrio raro es un error o un dato.

Y ahora la parte incómoda. Usá tu propia función para preguntar qué filas marca la regla
mirando **una columna por vez**, y quedate con las que pasan ese control.

In [ ]:
marcadas = set()
for col in ["population", "households", "total_rooms"]:
    marcadas |= set(detectar_outliers(df, col).index)

limpias = df.drop(index=marcadas)      # las que pasan el control columna por columna
caso = limpias.nlargest(1, "personas_por_hogar").iloc[0]

print(f"Filas que pasan el control en las tres columnas: {nro(len(limpias))}\n")
for col in ["population", "households", "total_rooms"]:
    pct = (df[col] < caso[col]).mean()
    print(f"  {col:12s} = {nro(caso[col]):>6}   percentil {pct:>3.0%} de su columna")
print(f"\n  ...y sin embargo: {caso['personas_por_hogar']:.1f} personas por hogar.")

**1.275 personas** es un barrio de tamaño perfectamente normal: percentil 56, el medio exacto
del dataset. **20 hogares** y **152 habitaciones** son valores bajos, pero la regla no los
marca — y no los marca *nunca*: con variables positivas y torcidas como éstas, `Q1 − 1,5·IQR`
da negativo, así que por abajo el límite no puede alcanzar a ningún dato.

Las tres columnas pasan el control de a una. La combinación —**64 personas por hogar**— es
imposible.

> **La regla:** el `k` es una decisión que tomás mirando la variable, no un default que se
> hereda. Y ninguna regla que mire una columna por vez puede ver una fila que sólo es absurda
> como combinación: eso se ve mirando el par, que es de lo que se trató toda esta práctica.

## ✍️ Para pensar

1. **Escribí un hallazgo de tres líneas** sobre alguna de las relaciones que viste acá — el
   par de siempre, el cruce de ingreso por segmento, o el que se te ocurra. El criterio es el
   de la clase:

   | línea | qué tiene que tener |
   |---|---|
   | 1 | que se entienda sin saber estadística |
   | 2 | el número **y** el `n` |
   | 3 | una razón **concreta** para dudar |

   La línea 3 es donde falla casi todo el mundo, y el modo típico de fallar es escribir una
   vaguedad (*"habría que investigar más"*) en lugar de una razón concreta.

2. **¿Qué `k` dejarías por defecto** en `detectar_outliers()` para `personas_por_hogar`: 1,5 o
   3,0? Elegí uno y escribí en una línea por qué. No hay respuesta correcta; hay respuestas
   justificadas y respuestas heredadas del default.

*Escribí tus respuestas acá (doble clic para editar esta celda).*

**1.**

**2.**